# 🧠 Curbo Guide Robot: Semantic NLP & Dense Vector Intent Engine
--------------------------------------------------------------------------------------
Welcome to the next-generation **Semantic NLP Intent Extraction Subsystem** for **Curbo**, the autonomous campus guide robot.

### 📌 Why Semantic Embeddings over Rule/Alias Matching?
In an engineering college campus, human queries are vast and non-deterministic:
* A student says: *"Where can I get printouts of my lab manual?"*
  * **Rule Matching / Fuzzy Ratio**: ❌ FAILS (Missing keyword "stationery" or "xerox").
  * **Semantic NLP Embeddings**: 🎯 **MATCHES `STATIONERY_XEROX`** with 89% semantic similarity!
* A visitor says: *"I feel dizzy and need a doctor."*
  * **Rule Matching**: ❌ FAILS.
  * **Semantic NLP Embeddings**: 🎯 **MATCHES `INFIRMARY_MEDICAL`** with 91% semantic similarity!

### 🔄 Multi-Tier Hybrid Architecture
```
┌──────────────────────────────────────────────────────────┐
│ Spoken Query (from Whisper STT / Web Mic)                │
└────────────────────────────┬─────────────────────────────┘
                             │
                             ▼
 ┌─────────────────────────────────────────────────────────┐
 │ Tier 1: Motion Controls (STOP, SLOW_DOWN, RESUME)       │
 └───────────────────────────┬─────────────────────────────┘
                             │ [If not motion command]
                             ▼
 ┌─────────────────────────────────────────────────────────┐
 │ Tier 2: Direct Alias Match (Exact/Fast Substring)       │
 └───────────────────────────┬─────────────────────────────┘
                             │ [If not exact alias]
                             ▼
 ┌─────────────────────────────────────────────────────────┐
 │ Tier 3: Dense Semantic Vector Embedding Search          │
 │ (Sentence-Transformers MiniLM-L6-v2 + Cosine Similarity) │
 └───────────────────────────┬─────────────────────────────┘
                             │
                             ▼
 ┌─────────────────────────────────────────────────────────┐
 │ Dispatched Waypoint ID (e.g., WP_FACILITY_XEROX_05)     │
 └─────────────────────────────────────────────────────────┘
```

## 📦 Step 1: Environment & Dependencies Setup
We load `sentence-transformers` for embedding generation, `torch` for tensor computations, `numpy` for vector math, and `rapidfuzz` for fast-path alias resolution.

In [ ]:
import os
import time
import numpy as np
import torch
from dataclasses import dataclass
from typing import Optional, Dict, Any, List, Tuple
from sentence_transformers import SentenceTransformer
from rapidfuzz import fuzz

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch running on: {device}")
print("Core libraries imported successfully.")

## 🗺️ Step 2: Semantic Knowledge Base (Descriptions + Aliases)
Unlike keyword dictionaries, each campus destination below includes:
1. **`aliases`**: Common short names and acronyms.
2. **`semantic_description`**: Functional descriptions detailing what services, actions, activities, or transactions take place there.

In [ ]:
CAMPUS_DESTINATIONS = {
    # 1. Administration & Governance
    "PRINCIPAL_OFFICE": {
        "canonical_name": "Principal's Office / Director Cabin",
        "aliases": ["principal office", "principal", "director office", "director cabin", "head of institution", "leadership office"],
        "semantic_description": "Office of the college principal and director. Administrative leadership, college head, meeting the director, executive administration, official appointments with head of institution.",
        "waypoint_id": "WP_ADMIN_PRINCIPAL_01",
        "confirmation_msg": "Leading you to the Principal's Office. Please follow me."
    },
    "DEAN_ACADEMICS": {
        "canonical_name": "Dean Academics Office",
        "aliases": ["dean office", "dean academics", "academic dean", "dean student affairs", "dean"],
        "semantic_description": "Dean of academic affairs, curriculum inquiries, academic syllabus, student council, grievance redressal, academic regulations and approvals.",
        "waypoint_id": "WP_ADMIN_DEAN_02",
        "confirmation_msg": "Guiding you to the Dean of Academics Office."
    },
    "ACCOUNTS_SECTION": {
        "canonical_name": "Accounts & Fee Counter",
        "aliases": ["accounts section", "accounts office", "fee counter", "fee payment", "challan counter", "finance department", "tuition fee desk", "cashier"],
        "semantic_description": "Pay tuition fees, college dues payment, challan submission, finance desk, account clearance, receipt collection, bus fees, scholarship disbursement, cashier.",
        "waypoint_id": "WP_ADMIN_ACCOUNTS_03",
        "confirmation_msg": "Taking you to the Accounts Section and Fee Counter."
    },
    "EXAM_CELL": {
        "canonical_name": "Exam Cell / Controller of Examinations",
        "aliases": ["exam cell", "examination cell", "exam branch", "exam department", "controller of examinations", "coe", "hall ticket counter", "exam hall", "marksheet desk"],
        "semantic_description": "Collect hall tickets, admit cards, semester exam registration, grade card, marksheets, re-evaluation forms, degree certificates, transcripts, controller of examinations.",
        "waypoint_id": "WP_ADMIN_EXAM_04",
        "confirmation_msg": "Sure! Taking you to the Exam Cell. Follow me."
    },
    "HR_OFFICE": {
        "canonical_name": "HR & Administrative Block",
        "aliases": ["hr office", "human resources", "hr department", "admin office", "administration block", "registrar office"],
        "semantic_description": "Staff human resources, employee onboarding, recruitment, administrative certificates, faculty records, service books, registrar administration.",
        "waypoint_id": "WP_ADMIN_HR_05",
        "confirmation_msg": "Understood. Guiding you to the HR & Administration Office."
    },
    "RECEPTION": {
        "canonical_name": "Reception / Front Desk",
        "aliases": ["reception", "front desk", "main entrance", "inquiry desk", "information desk", "help desk", "visitor lobby"],
        "semantic_description": "Main entrance lobby, visitor registration, guest check-in, general college inquiries, campus information desk, welcoming area.",
        "waypoint_id": "WP_ADMIN_RECEPTION_06",
        "confirmation_msg": "Leading you to the Reception now."
    },

    # 2. Academic Departments
    "CSE_DEPARTMENT": {
        "canonical_name": "Computer Science & Engineering (CSE)",
        "aliases": ["cse department", "computer science", "cse block", "it department", "information technology", "cse hod", "cse faculty"],
        "semantic_description": "Computer Science and Engineering department, IT branch, software faculty cabins, coding classrooms, computer science labs, CSE HOD cabin.",
        "waypoint_id": "WP_DEPT_CSE_01",
        "confirmation_msg": "Alright! Guiding you to the Computer Science & Engineering Department."
    },
    "ECE_DEPARTMENT": {
        "canonical_name": "Electronics & Communication (ECE)",
        "aliases": ["ece department", "electronics department", "ece block", "communication department", "ece hod", "ece faculty"],
        "semantic_description": "Electronics and Communication engineering department, signal processing, communication labs, circuit faculty, telecom department, ECE HOD.",
        "waypoint_id": "WP_DEPT_ECE_02",
        "confirmation_msg": "Heading towards the Electronics and Communication Department."
    },
    "EEE_DEPARTMENT": {
        "canonical_name": "Electrical & Electronics (EEE)",
        "aliases": ["eee department", "electrical department", "eee block", "electrical engineering", "eee hod"],
        "semantic_description": "Electrical and Electronics engineering department, electrical machines, power electronics, power grid circuits, EEE faculty and HOD.",
        "waypoint_id": "WP_DEPT_EEE_03",
        "confirmation_msg": "Leading you to the Electrical & Electronics Department."
    },
    "MECH_DEPARTMENT": {
        "canonical_name": "Mechanical Engineering",
        "aliases": ["mechanical department", "mech department", "mechanical block", "mech engineering", "mech hod"],
        "semantic_description": "Mechanical engineering department, thermodynamics, fluid mechanics, design faculty, automotive engineering, ME HOD cabin.",
        "waypoint_id": "WP_DEPT_MECH_04",
        "confirmation_msg": "Taking you to the Mechanical Engineering Department."
    },
    "CIVIL_DEPARTMENT": {
        "canonical_name": "Civil Engineering",
        "aliases": ["civil department", "civil engineering", "civil block", "civil hod"],
        "semantic_description": "Civil engineering department, structural engineering, geotechnical lab, surveying, construction materials, civil faculty cabins.",
        "waypoint_id": "WP_DEPT_CIVIL_05",
        "confirmation_msg": "Guiding you to the Civil Engineering Department."
    },
    "AIML_DEPARTMENT": {
        "canonical_name": "AI & Data Science (AIML/AIDS)",
        "aliases": ["ai department", "aiml department", "artificial intelligence", "data science department", "aids department", "aiml block"],
        "waypoint_id": "WP_DEPT_AIML_06",
        "confirmation_msg": "Leading you to the AI & Machine Learning Department."
    },
    "HOD_OFFICE": {
        "canonical_name": "HOD Cabin / Department Head",
        "aliases": ["hod office", "head of department", "hod cabin", "hod room", "hod", "h o d"],
        "semantic_description": "Head of department office, meeting the branch HOD, department head cabin, project approval signature, permission letter.",
        "waypoint_id": "WP_DEPT_HOD_07",
        "confirmation_msg": "Taking you to the HOD Office."
    },
    "MY_OFFICE": {
        "canonical_name": "Faculty / Staff Room",
        "aliases": ["my office", "faculty room", "staff cabin", "teacher office", "prof office", "staff room", "faculty cabins"],
        "semantic_description": "Professor cabins, meeting professors and lecturers, faculty room, staff room, assistant professors desk.",
        "waypoint_id": "WP_DEPT_FACULTY_08",
        "confirmation_msg": "Guiding you to the Faculty and Staff Office."
    },

    # 3. Laboratories & Workshops
    "CENTRAL_COMPUTING_FACILITY": {
        "canonical_name": "Central Computing Facility (CCF)",
        "aliases": ["central computer facility", "ccf", "computer center", "programming lab", "central lab", "software lab", "computer lab 1"],
        "semantic_description": "Central computer center, programming practicals, high speed internet systems, Linux lab, Python and Java coding lab, online exams.",
        "waypoint_id": "WP_LAB_CCF_01",
        "confirmation_msg": "Taking you to the Central Computing Facility."
    },
    "IOT_ROBOTICS_LAB": {
        "canonical_name": "IoT & Robotics Lab",
        "aliases": ["iot lab", "robotics lab", "embedded systems lab", "arduino lab", "microcontroller lab", "automation lab"],
        "semantic_description": "Internet of Things laboratory, robotics prototyping, Arduino and Raspberry Pi programming, microcontrollers, drone building, sensor interfacing.",
        "waypoint_id": "WP_LAB_ROBOTICS_02",
        "confirmation_msg": "Heading towards the IoT and Robotics Laboratory."
    },
    "VLSI_HARDWARE_LAB": {
        "canonical_name": "VLSI & Digital Electronics Lab",
        "aliases": ["vlsi lab", "electronics lab", "digital electronics lab", "dsp lab", "hardware lab", "circuit lab"],
        "semantic_description": "Chip design lab, Verilog and FPGA programming, digital circuit simulation, oscilloscope breadboard testing, hardware design.",
        "waypoint_id": "WP_LAB_VLSI_03",
        "confirmation_msg": "Guiding you to the VLSI and Hardware Lab."
    },
    "MECHANICAL_WORKSHOP": {
        "canonical_name": "Mechanical Workshop & CAD Lab",
        "aliases": ["mechanical workshop", "workshop", "cad lab", "cam lab", "lathe shop", "welding shop", "3d printing lab", "manufacturing lab"],
        "semantic_description": "Mechanical workshop, lathe machine turning, welding, carpentry, sheet metal work, 3D printing, CAD CAM modeling, machining.",
        "waypoint_id": "WP_LAB_WORKSHOP_04",
        "confirmation_msg": "Leading you to the Mechanical Engineering Workshop."
    },
    "PHYSICS_CHEMISTRY_LAB": {
        "canonical_name": "Physics & Chemistry Labs",
        "aliases": ["physics lab", "chemistry lab", "basic science lab", "first year lab"],
        "semantic_description": "First year engineering labs, optics experiments, titration, chemistry laboratory, physics apparatus experiments.",
        "waypoint_id": "WP_LAB_SCI_05",
        "confirmation_msg": "Taking you to the Science and Basic Engineering Laboratories."
    },

    # 4. Student Support & Facilities
    "LIBRARY": {
        "canonical_name": "Central Library & Digital Reading Room",
        "aliases": ["library", "central library", "reading room", "digital library", "book bank", "reference section", "study hall"],
        "semantic_description": "Borrow books, return library books, study quietly, reference journals, digital library computers, research papers, silent reading area.",
        "waypoint_id": "WP_FACILITY_LIB_01",
        "confirmation_msg": "Understood. Leading you to the Central Library."
    },
    "PLACEMENT_CELL": {
        "canonical_name": "Training & Placement (T&P) Cell",
        "aliases": ["placement cell", "training and placement", "tpo", "tpo office", "campus interview room", "placement officer", "internship desk"],
        "semantic_description": "Campus placement drives, job interviews, resume submission, recruitment companies, training and placement officer, internships, mock interviews.",
        "waypoint_id": "WP_FACILITY_TPO_02",
        "confirmation_msg": "Taking you to the Training and Placement Cell."
    },
    "AUDITORIUM": {
        "canonical_name": "Main Auditorium",
        "aliases": ["auditorium", "main auditorium", "cultural hall", "audi", "convention center", "amphitheater"],
        "waypoint_id": "WP_FACILITY_AUDI_03",
        "confirmation_msg": "Sure! Guiding you to the Main Auditorium."
    },
    "SEMINAR_HALL": {
        "canonical_name": "Seminar Hall / Conference Room",
        "aliases": ["seminar hall", "seminar hall a", "seminar hall b", "conference room", "presentation hall", "board room"],
        "semantic_description": "Guest lectures, technical seminars, project presentations, department workshops, paper presentations, academic conferences.",
        "waypoint_id": "WP_FACILITY_SEMINAR_04",
        "confirmation_msg": "Leading you to the Seminar Hall."
    },
    "STATIONERY_XEROX": {
        "canonical_name": "Stationery & Xerox / Photocopy Shop",
        "aliases": ["xerox shop", "photocopy center", "stationery", "printout shop", "xerox", "printing counter", "spiral binding"],
        "semantic_description": "Photocopy documents, printout lab manuals and project reports, Xerox copies, buy pens, notebooks, graph papers, spiral binding.",
        "waypoint_id": "WP_FACILITY_XEROX_05",
        "confirmation_msg": "Taking you to the Stationery and Photocopy Center."
    },
    "INFIRMARY_MEDICAL": {
        "canonical_name": "First Aid & Medical Dispensary",
        "aliases": ["medical room", "dispensary", "first aid", "health center", "infirmary", "doctor room", "emergency care"],
        "semantic_description": "Feeling sick, headache, fever, first aid treatment, doctor consultation, band-aid, medical emergency, resting bed, medicine dispensary.",
        "waypoint_id": "WP_FACILITY_MED_06",
        "confirmation_msg": "Guiding you to the Medical and First Aid Room."
    },
    "RESTROOMS": {
        "canonical_name": "Restrooms / Washrooms",
        "aliases": ["restroom", "restrooms", "washroom", "washrooms", "toilet", "toilets", "ladies washroom", "gents washroom"],
        "semantic_description": "Washrooms, toilets, restrooms, hand wash, urinals, ladies and gents washroom facility.",
        "waypoint_id": "WP_FACILITY_WASHROOM_07",
        "confirmation_msg": "Leading you to the nearest Restrooms."
    },

    # 5. Amenities & Campus Living
    "CAFETERIA": {
        "canonical_name": "Cafeteria / Canteen",
        "aliases": ["cafeteria", "canteen", "food court", "cafe", "mess", "refreshments", "snacks", "nescafe counter"],
        "semantic_description": "Food court, eat lunch, snacks, coffee, tea, breakfast, drinking water, college canteen, eating meal.",
        "waypoint_id": "WP_AMENITY_CANTEEN_01",
        "confirmation_msg": "Alright! Let's head to the Cafeteria."
    },
    "SPORTS_COMPLEX": {
        "canonical_name": "Sports Complex & Gymnasium",
        "aliases": ["sports room", "gym", "gymnasium", "sports complex", "indoor stadium", "badminton court", "sports ground"],
        "semantic_description": "Play sports, gym workout, fitness center, indoor games, table tennis, badminton court, basketball, cricket kit.",
        "waypoint_id": "WP_AMENITY_SPORTS_02",
        "confirmation_msg": "Guiding you to the Sports Complex and Gym."
    },
    "HOSTELS": {
        "canonical_name": "Campus Hostels",
        "aliases": ["hostel", "hostels", "boys hostel", "girls hostel", "warden office", "hostel mess"],
        "semantic_description": "Student accommodation, residential rooms, boys hostel, girls hostel, hostel warden cabin, hostel mess.",
        "waypoint_id": "WP_AMENITY_HOSTEL_03",
        "confirmation_msg": "Taking you towards the Campus Hostels."
    },
    "SECURITY_MAIN_GATE": {
        "canonical_name": "Main Gate & Security Desk",
        "aliases": ["main gate", "security cabin", "security gate", "entrance gate", "visitor desk", "parking", "parking lot"],
        "semantic_description": "Campus entrance and exit, vehicle parking lot, security guard cabin, visitor pass check, gate pass.",
        "waypoint_id": "WP_AMENITY_SECURITY_04",
        "confirmation_msg": "Leading you to the Main Gate and Security Desk."
    }
}

MOTION_COMMANDS = {
    "STOP": ["stop", "halt", "freeze", "stop here", "wait"],
    "SLOW_DOWN": ["slow down", "slower", "too fast", "walk slower", "wait for me"],
    "SPEED_UP": ["speed up", "faster", "go faster", "hurry"],
    "RESUME": ["continue", "keep going", "resume", "let's go"],
    "CANCEL": ["cancel", "nevermind", "abort", "go back"]
}

print(f"Loaded {len(CAMPUS_DESTINATIONS)} semantic campus destinations with descriptions and aliases.")

## ⚡ Step 3: Vector Embedding & Indexing Engine
We use `all-MiniLM-L6-v2`:
* **Model Size**: ~30 MB (Super lightweight, runs in milliseconds on Raspberry Pi 5 / CPU).
* **Embedding Dimension**: 384 dimensions.
* **Pre-computation**: We compute embeddings for all campus destinations **once at startup** and normalize them for instant dot-product cosine similarity calculation ($O(1)$ matrix multiplication).

In [ ]:
class CampusSemanticIndex:
    """Pre-computes and holds normalized vector embeddings for all campus destinations."""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        print(f"[SemanticIndex] Loading SentenceTransformer '{model_name}' on {device}...")
        self.model = SentenceTransformer(model_name, device=device)
        self.keys: List[str] = list(CAMPUS_DESTINATIONS.keys())
        
        # Build comprehensive semantic corpus for each destination (combining name, aliases, and description)
        self.corpus_texts = []
        for k in self.keys:
            data = CAMPUS_DESTINATIONS[k]
            aliases_str = ", ".join(data["aliases"])
            corpus_entry = f"{data['canonical_name']}. Aliases: {aliases_str}. Description: {data['semantic_description']}"
            self.corpus_texts.append(corpus_entry)
        
        # Encode and L2-normalize for fast cosine similarity via dot product
        t0 = time.time()
        raw_embeddings = self.model.encode(self.corpus_texts, convert_to_numpy=True, show_progress_bar=False)
        # Normalize vectors: ||v|| = 1.0
        norms = np.linalg.norm(raw_embeddings, axis=1, keepdims=True)
        self.embeddings = raw_embeddings / np.maximum(norms, 1e-9)
        encode_time = (time.time() - t0) * 1000
        
        print(f"[SemanticIndex] Indexed {len(self.keys)} destinations in {encode_time:.2f}ms (Embedding Shape: {self.embeddings.shape})")

    def search(self, query: str, top_k: int = 3) -> List[Tuple[str, float]]:
        """Encodes user query and computes cosine similarity with all destinations."""
        if not query.strip():
            return []
        
        query_vec = self.model.encode([query], convert_to_numpy=True)
        query_norm = query_vec / np.maximum(np.linalg.norm(query_vec), 1e-9)
        
        # Dot product of normalized vectors equals cosine similarity
        similarities = np.dot(self.embeddings, query_norm.T).flatten()
        
        # Rank by score descending
        top_indices = np.argsort(similarities)[::-1][:top_k]
        results = [(self.keys[idx], float(similarities[idx])) for idx in top_indices]
        return results

semantic_index = CampusSemanticIndex()
print("Semantic Index ready.")